# Milestone 1 — Thu thập dữ liệu

## Mục tiêu và thiết kế

M1 tạo corpus làm việc cho parser: metadata của paper, nguồn LaTeX/BibTeX của từng
revision và metadata references từ Semantic Scholar. Đơn vị tải nguồn là một version;
đơn vị tổng hợp metadata và references là một paper.

Pipeline gồm bốn bước: **ID được giao → metadata/version → nguồn và references →
audit, checkpoint và hai bảng M1**. Chương 1–4 giải thích từng bước; chương 5 trình
bày phép đo crawler. Cách phân chia này tách phương pháp khỏi kết quả thực nghiệm.

## Dữ liệu đánh giá

Báo cáo dùng các lô backup Drive hiện có đã kiểm tra checksum. Đây là tập khôi phục
một phần, chọn theo giới hạn kích thước archive, không phải mẫu ngẫu nhiên. Counter
checkpoint và số paper có nguồn được audit là hai đại lượng khác nhau.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
display(Markdown('**Bảng M1.1. Phạm vi dữ liệu và đơn vị thống kê.**'))
display(pd.DataFrame([
    ('ID được giao cho nhóm', scope['group_assigned'], 'paper'),
    ('ID được giao cho một thành viên', scope['member_assigned'], 'paper'),
    ('Snapshot thực sự được đánh giá', scope['papers'], 'paper'),
    ('Các revision thuộc snapshot', scope['versions'], 'version'),
    ('Lô backup dùng để dựng snapshot', scope['archives_used'], 'archive'),
    ('Counter checkpoint, chưa audit toàn chuỗi', scope['checkpoint_reported_ids'], 'ID'),
], columns=['Phạm vi', 'Số lượng', 'Đơn vị']))
display(Markdown(f"Snapshot chiếm **{scope['papers']/scope['group_assigned']:.3%}** dải của nhóm. "
                 'Mọi tỷ lệ dưới đây chỉ mô tả tập đã kiểm tra.'))

## Kết quả chính

“Nguồn hợp lệ” gồm TeX đã xác minh và trường hợp không có LaTeX có manifest làm bằng
chứng. Tỷ lệ có kết quả S2 dùng cả `success` và `empty` được xác nhận. Paper đủ nguồn
và S2 được tính bằng giao của hai tập; không lấy tỷ lệ nguồn thay cho tỷ lệ thu thập
đủ cả hai thành phần. Ngày version còn thiếu được thống kê riêng ở chương 1.

In [ ]:
metrics = pd.DataFrame([
    ('Version có TeX đã xác minh', m1['source_audit'].get('tex_verified', 0), scope['versions']),
    ('Version có nguồn hợp lệ, gồm empty', m1['source_verified'], scope['versions']),
    ('Paper có mọi nguồn hợp lệ', m1['papers_with_all_sources_verified'], scope['papers']),
    ('Paper có S2 success/empty', m1['verified_reference_papers'], scope['papers']),
    ('Paper có mọi nguồn hợp lệ và S2', m1['papers_with_sources_and_s2_verified'], scope['papers']),
], columns=['Chỉ tiêu', 'Số đạt', 'Mẫu số'])
metrics['Tỷ lệ (%)'] = (100 * metrics['Số đạt'] / metrics['Mẫu số']).round(2)
display(Markdown('**Bảng M1.2. Kết quả audit trên snapshot; mẫu số ghi riêng cho từng chỉ tiêu.**'))
display(metrics)
plot = metrics.iloc[2:].copy()
ax = plot.plot.barh(x='Chỉ tiêu', y='Tỷ lệ (%)', legend=False, xlim=(0, 112), color='#2f628e')
for i, row in enumerate(plot.to_dict('records')):
    ax.text(row['Tỷ lệ (%)'] + 1, i, f"{row['Số đạt']}/{row['Mẫu số']}", va='center')
ax.set_xlabel('Tỷ lệ paper trong snapshot (%)'); ax.set_ylabel('')
plt.tight_layout(); plt.show()
display(Markdown('**Hình M1.1. Độ phủ nguồn và references.** Các thanh có cùng mẫu số paper; '
                 'chênh lệch cho thấy phần references cần được bổ sung.'))

In [ ]:
display(Markdown('**Bảng M1.3. Kích thước và references, kèm tập đủ điều kiện.**'))
display(pd.DataFrame([
    ('Byte trung bình trước lọc', f"{m1['mean_before_bytes']:,.2f}", f"{m1['eligible_size_papers']} paper đủ nguồn TeX"),
    ('Byte trung bình sau lọc', f"{m1['mean_after_bytes']:,.2f}", f"Cùng {m1['eligible_size_papers']} paper"),
    ('Mức giảm byte do lọc', f"{m1['source_reduction']:.2%}", 'Tỷ số hai tổng trên cùng tập'),
    ('References trung bình / paper', f"{m1['mean_references_verified_papers']:.2f}", f"{m1['verified_reference_papers']} paper có S2 success/empty"),
    ('Cạnh references làm việc', str(m1['working_reference_rows']), 'Các cạnh thực có trong snapshot'),
    ('Cạnh có đích arXiv xuất bảng chuẩn', str(m1['arxiv_reference_rows']), 'Tập con của working references'),
], columns=['Chỉ tiêu', 'Giá trị', 'Tập tính']))

## Nhận xét

Tải và lọc nguồn tạo được đầu vào để tiếp tục M2, còn độ phủ references là hạn chế chính
của M1 trên snapshot. Việc thiếu DOI, venue hoặc ngày revision cần được diễn giải theo
từng trường, thay vì gộp thành một nhãn “thành công”. Phép đo Colab đã lưu được báo cáo
riêng theo đúng 14 paper xử lý trong lượt đó ở [chương 5](05_performance_benchmark.ipynb).

Tiếp tục đọc [discovery](01_discovery_and_versions.ipynb), [nguồn và kích thước](02_source_download_and_cleaning.ipynb),
[references](03_reference_extraction.ipynb) và [schema/validation](04_schema_and_data_quality.ipynb).